In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pretty_midi
import os, sys, math, random, zipfile
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


CSV_PATH   = '/content/maestro/maestro-v3.0.0/maestro-v3.0.0.csv'
MIDI_ROOT  = '/content/maestro/maestro-v3.0.0'
OUT_DIR    = '/content/drive/MyDrive/music-project/outputs/generated_midis'
PLOT_DIR   = '/content/drive/MyDrive/music-project/outputs/plots'
SURVEY_DIR = '/content/drive/MyDrive/music-project/outputs/survey_results'
TOKEN_DIR  = '/content/drive/MyDrive/music-project/outputs/survey_results/tokens'
WEIGHTS    = '/content/drive/MyDrive/music-project/outputs/transformer_weights.pt'

for d in [OUT_DIR, PLOT_DIR, SURVEY_DIR, TOKEN_DIR]:
    os.makedirs(d, exist_ok=True)
print("All folders ready!")


zip_path = '/content/drive/MyDrive/Music Project/maestro-v3.0.0-midi.zip'
if not os.path.exists(MIDI_ROOT):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/maestro/')
    print("Dataset extracted!")
else:
    print("Dataset already extracted!")


NOTE_ON_OFFSET    = 0
NOTE_OFF_OFFSET   = 128
TIME_SHIFT_OFFSET = 256
VELOCITY_OFFSET   = 356
PAD_TOKEN         = 388
EOS_TOKEN         = 389
VOCAB_SIZE        = 390
SPECIAL_TOKENS    = {PAD_TOKEN, EOS_TOKEN}

BAROQUE_C   = ['Bach','Handel','Scarlatti','Vivaldi','Telemann']
CLASSICAL_C = ['Mozart','Haydn','Clementi','Dussek']
ROMANTIC_C  = ['Chopin','Liszt','Brahms','Schumann','Schubert','Mendelssohn','Tchaikovsky']
GENRE_NAMES = {0:'Baroque', 1:'Classical', 2:'Romantic', 3:'Modern'}
NUM_GENRES  = 4

SEQ_LEN     = 128
D_MODEL     = 128
NHEAD       = 4
NUM_LAYERS  = 2
DIM_FF      = 256
DROPOUT     = 0.1
EPOCHS      = 3
BATCH_SIZE  = 8
MAX_FILES   = 10
LR          = 3e-4
GEN_LENGTH  = 256
TOP_K       = 40
TEMPERATURE = 1.0
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE} | Epochs: {EPOCHS} | MAX_FILES: {MAX_FILES} | SEQ_LEN: {SEQ_LEN}")


def midi_to_tokens(path, max_tokens=1024):
    try:
        pm = pretty_midi.PrettyMIDI(path)
    except Exception:
        return []
    events = []
    for inst in pm.instruments:
        for note in inst.notes:
            events.append((note.start, 'on',  note.pitch, note.velocity))
            events.append((note.end,   'off', note.pitch, 0))
    events.sort(key=lambda e: e[0])
    tokens, prev = [], 0.0
    for t, kind, pitch, vel in events:
        bins = min(int((t - prev) / 0.01), 99)
        if bins > 0:
            tokens.append(TIME_SHIFT_OFFSET + bins)
        if kind == 'on':
            tokens.append(VELOCITY_OFFSET + min(int(vel / 4), 31))
            tokens.append(NOTE_ON_OFFSET + pitch)
        else:
            tokens.append(NOTE_OFF_OFFSET + pitch)
        prev = t
    tokens.append(EOS_TOKEN)
    return tokens[:max_tokens]


def tokens_to_midi(tokens, out_path):
    pm, piano = pretty_midi.PrettyMIDI(), pretty_midi.Instrument(program=0)
    t, vel, active = 0.0, 64, {}
    for tok in tokens:
        if tok in SPECIAL_TOKENS:      continue
        elif tok >= VELOCITY_OFFSET:   vel = ((tok - VELOCITY_OFFSET) + 1) * 4
        elif tok >= TIME_SHIFT_OFFSET: t  += (tok - TIME_SHIFT_OFFSET) * 0.01
        elif tok >= NOTE_OFF_OFFSET:
            pitch = tok - NOTE_OFF_OFFSET
            if pitch in active:
                piano.notes.append(pretty_midi.Note(
                    active[pitch][1], pitch, active[pitch][0],
                    max(active[pitch][0] + 0.05, t)))
                del active[pitch]
        else:
            active[tok - NOTE_ON_OFFSET] = (t, vel)
    for pitch, (s, v) in active.items():
        piano.notes.append(pretty_midi.Note(v, pitch, s, s + 0.5))
    pm.instruments.append(piano)
    pm.write(out_path)
    return len(piano.notes), pm.get_end_time()


def composer_to_genre(name):
    name = str(name).lower()
    for c in BAROQUE_C:
        if c.lower() in name: return 0
    for c in CLASSICAL_C:
        if c.lower() in name: return 1
    for c in ROMANTIC_C:
        if c.lower() in name: return 2
    return 3


class MAESTRODataset(Dataset):
    def __init__(self, split, seq_len=SEQ_LEN, max_files=MAX_FILES, save_tokens=False):
        self.seq_len = seq_len
        self.seqs, self.genres = [], []
        df = pd.read_csv(CSV_PATH)
        df = df[df['split'] == split].head(max_files)
        df['canonical_composer'] = df.get('canonical_composer', '').fillna('')
        for _, row in df.iterrows():
            path = os.path.join(MIDI_ROOT, row['midi_filename'])
            if not os.path.exists(path): continue
            genre  = composer_to_genre(row['canonical_composer'])
            tokens = midi_to_tokens(path, max_tokens=seq_len * 4)
            if len(tokens) < seq_len + 1: continue
            if save_tokens:
                fname = os.path.basename(row['midi_filename']).replace('.midi','.pt').replace('.mid','.pt')
                torch.save(torch.tensor(tokens[:seq_len], dtype=torch.long),
                           os.path.join(TOKEN_DIR, fname))
            for s in range(0, len(tokens) - seq_len, seq_len):
                self.seqs.append(tokens[s:s + seq_len])
                self.genres.append(genre)
        counts = {}
        for g in self.genres:
            counts[GENRE_NAMES[g]] = counts.get(GENRE_NAMES[g], 0) + 1
        print(f"[{split}] {len(self.seqs)} sequences | {counts}")

    def __len__(self): return len(self.seqs)
    def __getitem__(self, idx):
        t = torch.tensor(self.seqs[idx],   dtype=torch.long)
        g = torch.tensor(self.genres[idx], dtype=torch.long)
        return t[:-1], t[1:], g


class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=2048, drop=0.1):
        super().__init__()
        self.drop = nn.Dropout(drop)
        pe  = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class MusicTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb   = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD_TOKEN)
        self.genre_emb = nn.Embedding(NUM_GENRES, D_MODEL)
        self.pos_enc   = PositionalEncoding(D_MODEL, max_len=SEQ_LEN+1, drop=DROPOUT)
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NHEAD, dim_feedforward=DIM_FF,
            dropout=DROPOUT, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=NUM_LAYERS)
        self.fc_out = nn.Linear(D_MODEL, VOCAB_SIZE)
        self._init()
        self.fc_out.weight = self.tok_emb.weight

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.01)

    def forward(self, x, genre):
        T    = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device).bool()
        emb  = self.pos_enc(self.tok_emb(x) * math.sqrt(D_MODEL)
                            + self.genre_emb(genre).unsqueeze(1))
        return self.fc_out(self.transformer(emb, mask=mask, is_causal=True))

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def compute_ppl(loss):
    return math.exp(min(loss, 20))


def train_model(model, train_loader, val_loader):
    model.to(DEVICE)
    opt  = optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, reduction='mean')
    hist = {'train_loss':[], 'val_loss':[], 'train_ppl':[], 'val_ppl':[]}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        eloss, nb = 0.0, 0
        for x, y, g in train_loader:
            x, y, g = x.to(DEVICE), y.to(DEVICE), g.to(DEVICE)
            logits  = model(x, g)
            loss    = crit(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            eloss += loss.item(); nb += 1
        avg_t = eloss / max(nb, 1)
        sch.step()

        model.eval()
        vloss, nvb = 0.0, 0
        with torch.no_grad():
            for x, y, g in val_loader:
                x, y, g = x.to(DEVICE), y.to(DEVICE), g.to(DEVICE)
                vloss += crit(model(x, g).reshape(-1, VOCAB_SIZE), y.reshape(-1)).item()
                nvb   += 1
        avg_v = vloss / max(nvb, 1)

        hist['train_loss'].append(avg_t)
        hist['val_loss'].append(avg_v)
        hist['train_ppl'].append(compute_ppl(avg_t))
        hist['val_ppl'].append(compute_ppl(avg_v))
        print(f"Epoch {epoch}/{EPOCHS}  Train Loss={avg_t:.4f} PPL={compute_ppl(avg_t):.1f}  Val Loss={avg_v:.4f} PPL={compute_ppl(avg_v):.1f}", flush=True)
    return hist


@torch.no_grad()
def generate(model, genre_id, length=GEN_LENGTH, temperature=TEMPERATURE, top_k=TOP_K):
    model.eval()
    tokens = [NOTE_ON_OFFSET + 60]
    genre  = torch.tensor([genre_id], device=DEVICE)
    for _ in range(length - 1):
        inp    = torch.tensor([tokens[-SEQ_LEN:]], dtype=torch.long, device=DEVICE)
        logits = model(inp, genre)[:, -1, :].clone()
        for s in SPECIAL_TOKENS: logits[0, s] = float('-inf')
        logits = logits / max(temperature, 1e-6)
        tv, ti = logits.topk(top_k)
        probs  = torch.softmax(tv, dim=-1)
        nxt    = ti[0, torch.multinomial(probs, 1)].item()
        tokens.append(nxt)
        if nxt == EOS_TOKEN: break
    return tokens


def verify_midi(path, min_notes=15, min_dur=2.0):
    try:
        pm  = pretty_midi.PrettyMIDI(path)
        n   = sum(len(i.notes) for i in pm.instruments)
        dur = pm.get_end_time()
        ok  = n >= min_notes and dur >= min_dur
        if not ok:
            print(f"FAIL: {os.path.basename(path)} — {n} notes, {dur:.1f}s")
        return ok
    except Exception as e:
        print(f"FAIL: {e}"); return False


def generate_all(model, n=10):
    saved, attempts = [], 0
    genres = [0, 1, 2, 3] * 4
    while len(saved) < n and attempts < n * 5:
        gid  = genres[len(saved) % len(genres)]
        path = f'{OUT_DIR}/task3_{GENRE_NAMES[gid]}_{len(saved)+1}.mid'
        toks = generate(model, gid)
        tokens_to_midi(toks, path)
        if verify_midi(path):
            saved.append(path)
            print(f"[{len(saved)}/{n}] {os.path.basename(path)}", flush=True)
        else:
            if os.path.exists(path): os.remove(path)
        attempts += 1
    print(f"Generated {len(saved)}/{n} valid compositions.")
    return saved


def rhythm_diversity(path, qms=50.0):
    pm  = pretty_midi.PrettyMIDI(path); dur = []
    for inst in pm.instruments:
        for n in inst.notes:
            dur.append(round((n.end - n.start)*1000/qms)*qms)
    return len(set(dur)) / len(dur) if dur else 0.0


def repetition_ratio(path, n=4):
    pm      = pretty_midi.PrettyMIDI(path)
    notes   = sorted([note for inst in pm.instruments for note in inst.notes], key=lambda x: x.start)
    pitches = [n.pitch for n in notes]
    if len(pitches) < n: return 0.0
    grams   = [tuple(pitches[i:i+n]) for i in range(len(pitches)-n+1)]
    counts  = Counter(grams)
    return sum(1 for c in counts.values() if c > 1) / len(grams)


def gen_random_midi(path, n_notes=80, dur_sec=20.0):
    pm, piano = pretty_midi.PrettyMIDI(), pretty_midi.Instrument(program=0)
    onsets = sorted(random.uniform(0, dur_sec-1) for _ in range(n_notes))
    for onset in onsets:
        d = random.choice([0.25, 0.5, 1.0])
        piano.notes.append(pretty_midi.Note(
            random.randint(40,100), random.randint(21,108),
            onset, min(onset+d, dur_sec)))
    pm.instruments.append(piano); pm.write(path)
    return len(piano.notes), pm.get_end_time()


class MarkovModel:
    def __init__(self):
        self.trans = defaultdict(lambda: defaultdict(int))
        self.durs, self.starts = [], []

    def fit(self, paths):
        for path in paths:
            try:
                pm    = pretty_midi.PrettyMIDI(path)
                notes = sorted([n for i in pm.instruments for n in i.notes], key=lambda n: n.start)
                if len(notes) < 2: continue
                self.starts.append(notes[0].pitch)
                for i in range(len(notes)-1):
                    self.trans[notes[i].pitch][notes[i+1].pitch] += 1
                    self.durs.append(notes[i].end - notes[i].start)
                self.durs.append(notes[-1].end - notes[-1].start)
            except: continue

    def _next(self, cur):
        c = self.trans.get(cur, {})
        if not c: return random.randint(21, 108)
        ps = list(c.keys()); tot = sum(c.values())
        return random.choices(ps, [c[p]/tot for p in ps])[0]

    def generate(self, path, n=150):
        pm, piano = pretty_midi.PrettyMIDI(), pretty_midi.Instrument(program=0)
        cur = random.choice(self.starts) if self.starts else 60
        t   = 0.0
        for _ in range(n):
            d = max(0.1, random.choice(self.durs) if self.durs else 0.5)
            piano.notes.append(pretty_midi.Note(random.randint(50,100), cur, t, t+d))
            t += 0.2; cur = self._next(cur)
        pm.instruments.append(piano); pm.write(path)
        return len(piano.notes), pm.get_end_time()

    def perplexity(self, paths):
        lp = []
        for path in paths:
            try:
                pm    = pretty_midi.PrettyMIDI(path)
                notes = sorted([n for i in pm.instruments for n in i.notes], key=lambda n: n.start)
                for i in range(len(notes)-1):
                    c = self.trans.get(notes[i].pitch, {}); tot = sum(c.values())
                    if tot == 0: continue
                    p = c.get(notes[i+1].pitch, 0) / tot
                    if p > 0: lp.append(math.log(p))
            except: continue
        return math.exp(-sum(lp)/len(lp)) if lp else float('inf')


def run_baselines(train_paths, val_paths, n=3):
    results = {}
    rds, rrs = [], []
    for i in range(n):
        p = f'{OUT_DIR}/baseline_random_{i+1}.mid'
        gen_random_midi(p); rds.append(rhythm_diversity(p)); rrs.append(repetition_ratio(p))
    results['Random Generator'] = {'perplexity':'N/A',
        'rhythm_div':f"{np.mean(rds):.3f}", 'rep_ratio':f"{np.mean(rrs):.3f}", 'genre_ctrl':'No'}
    mk = MarkovModel(); mk.fit(train_paths[:10])
    mds, mrs = [], []
    for i in range(n):
        p = f'{OUT_DIR}/baseline_markov_{i+1}.mid'
        mk.generate(p); mds.append(rhythm_diversity(p)); mrs.append(repetition_ratio(p))
    ppl = mk.perplexity(val_paths[:5])
    results['Markov Chain'] = {'perplexity':f"{ppl:.1f}",
        'rhythm_div':f"{np.mean(mds):.3f}", 'rep_ratio':f"{np.mean(mrs):.3f}", 'genre_ctrl':'No'}
    return results


def plot_metrics(hist):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(hist['train_loss'], marker='o', label='Train',      linewidth=2)
    a1.plot(hist['val_loss'],   marker='s', label='Validation', linewidth=2, linestyle='--')
    a1.set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Task 3 — NLL Loss')
    a1.legend(); a1.grid(alpha=0.3)
    a2.plot(hist['train_ppl'],  marker='o', label='Train PPL',  linewidth=2)
    a2.plot(hist['val_ppl'],    marker='s', label='Val PPL',    linewidth=2, linestyle='--')
    a2.set(xlabel='Epoch', ylabel='Perplexity', title='Task 3 — Perplexity')
    a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout()
    path = f'{PLOT_DIR}/metrics_task3.png'
    plt.savefig(path, dpi=150); plt.close()
    print(f"Plot saved: {path}")


def plot_comparison(baseline_results, transformer_rd, transformer_rr):
    all_models = {**baseline_results,
                  'Transformer': {'rhythm_div': f'{transformer_rd:.3f}',
                                  'rep_ratio':  f'{transformer_rr:.3f}'}}
    names = list(all_models.keys())
    rd    = [float(m['rhythm_div']) for m in all_models.values()]
    rr    = [float(m['rep_ratio'])  for m in all_models.values()]
    x, w  = np.arange(len(names)), 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x-w/2, rd, w, label='Rhythm Diversity', color='steelblue', alpha=0.85)
    ax.bar(x+w/2, rr, w, label='Repetition Ratio', color='salmon',    alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylabel('Score'); ax.set_title('Baseline vs Transformer Comparison')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = f'{PLOT_DIR}/comparison_task3.png'
    plt.savefig(path, dpi=150); plt.close()
    print(f"Plot saved: {path}")


print("Loading dataset...")
train_ds = MAESTRODataset('train',      save_tokens=True)
val_ds   = MAESTRODataset('validation', save_tokens=False)

if len(train_ds) == 0:
    print("ERROR: No training sequences found.")
    sys.exit(0)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("Dataset ready!")

model = MusicTransformer()
print("=" * 55)
print("ARCHITECTURE")
print("=" * 55)
print(model)
print(f"\nTotal trainable parameters: {model.n_params():,}")
print(f"d_model={D_MODEL} | nhead={NHEAD} | layers={NUM_LAYERS} | ff={DIM_FF} | seq_len={SEQ_LEN}")
print("=" * 55)

print("\nTraining...")
history = train_model(model, train_loader, val_loader)
torch.save(model.state_dict(), WEIGHTS)
print(f"Model saved: {WEIGHTS}")

plot_metrics(history)

print("\nGenerating 10 MIDI compositions...")
saved_paths = generate_all(model, n=10)
print("MIDI files saved!")

print("\nMetrics on generated samples:")
sample_metrics = []
for path in saved_paths:
    try:
        rd = rhythm_diversity(path)
        rr = repetition_ratio(path)
        sample_metrics.append((os.path.basename(path), rd, rr))
        print(f"  {os.path.basename(path):<42}  RhyDiv={rd:.3f}  RepRatio={rr:.3f}")
    except Exception as e:
        print(f"  Metrics error: {e}")

print("\nRunning baselines...")
df_all = pd.read_csv(CSV_PATH)
train_paths = [os.path.join(MIDI_ROOT, r['midi_filename'])
               for _, r in df_all[df_all['split']=='train'].head(MAX_FILES).iterrows()
               if os.path.exists(os.path.join(MIDI_ROOT, r['midi_filename']))]
val_paths   = [os.path.join(MIDI_ROOT, r['midi_filename'])
               for _, r in df_all[df_all['split']=='validation'].head(10).iterrows()
               if os.path.exists(os.path.join(MIDI_ROOT, r['midi_filename']))]

baseline_results = run_baselines(train_paths, val_paths)
print("Baselines done!")

if sample_metrics:
    avg_rd = float(np.mean([m[1] for m in sample_metrics]))
    avg_rr = float(np.mean([m[2] for m in sample_metrics]))
    plot_comparison(baseline_results, avg_rd, avg_rr)

print("\n" + "=" * 55)
print("TASK 3 RESULTS")
print("=" * 55)
print(f"Final Train Loss      : {history['train_loss'][-1]:.4f}")
print(f"Final Val Loss        : {history['val_loss'][-1]:.4f}")
print(f"Final Train PPL       : {history['train_ppl'][-1]:.2f}")
print(f"Final Val Perplexity  : {history['val_ppl'][-1]:.2f}")
print(f"Compositions saved    : {len(saved_paths)}/10")
print(f"Plots saved to        : {PLOT_DIR}")
print(f"Weights saved to      : {WEIGHTS}")
print("=" * 55)

print("\nPlaying generated MIDI files:")
from IPython.display import Audio, display
import glob

midi_files = sorted(glob.glob(f'{OUT_DIR}/task3_*.mid'))
for f in midi_files[:3]:
    print(f"► {os.path.basename(f)}")
    display(Audio(f))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All folders ready!
Dataset already extracted!
Device: cpu | Epochs: 3 | MAX_FILES: 10 | SEQ_LEN: 128
Loading dataset...
[train] 30 sequences | {'Modern': 30}
[validation] 30 sequences | {'Modern': 30}
Dataset ready!
ARCHITECTURE
MusicTransformer(
  (tok_emb): Embedding(390, 128, padding_idx=388)
  (genre_emb): Embedding(4, 128)
  (pos_enc): PositionalEncoding(
    (drop): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm

/tmp/ipykernel_4633/627991603.py:190: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=NUM_LAYERS)


Epoch 1/3  Train Loss=5.8599 PPL=350.7  Val Loss=5.6770 PPL=292.1
Epoch 2/3  Train Loss=5.5565 PPL=258.9  Val Loss=5.4004 PPL=221.5
Epoch 3/3  Train Loss=5.3363 PPL=207.7  Val Loss=5.3017 PPL=200.7
Model saved: /content/drive/MyDrive/music-project/outputs/transformer_weights.pt
Plot saved: /content/drive/MyDrive/music-project/outputs/plots/metrics_task3.png

Generating 10 MIDI compositions...
[1/10] task3_Baroque_1.mid
[2/10] task3_Classical_2.mid
[3/10] task3_Romantic_3.mid
[4/10] task3_Modern_4.mid
[5/10] task3_Baroque_5.mid
[6/10] task3_Classical_6.mid
[7/10] task3_Romantic_7.mid
[8/10] task3_Modern_8.mid
[9/10] task3_Baroque_9.mid
[10/10] task3_Classical_10.mid
Generated 10/10 valid compositions.
MIDI files saved!

Metrics on generated samples:
  task3_Baroque_1.mid                         RhyDiv=0.400  RepRatio=0.000
  task3_Classical_2.mid                       RhyDiv=0.318  RepRatio=0.000
  task3_Romantic_3.mid                        RhyDiv=0.421  RepRatio=0.000
  task3_Modern_4

► task3_Baroque_5.mid


► task3_Baroque_9.mid
